# Remote Work and Urban Housing Prices: Exploratory Data Analysis
Team 10 — Sai Krishna Varshith Danda, Gayathri Kota, Eshitha Chowdary Kaka, Zain Khan

*Skeleton. Section 1 pulls its tables from the pipeline's data-quality reports so the text and the data cannot drift apart; Section 2 is the figure plan, with the question each figure answers. Replace each TODO with a figure and a short interpretation paragraph.*

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN, LOGS = ROOT / "data" / "clean", ROOT / "outputs" / "logs"
FIGS = ROOT / "outputs" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
TABLES = ROOT / "outputs" / "tables"; TABLES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))   # viz.py, viz_eda.py
warnings.filterwarnings("ignore", category=FutureWarning)

def log(name):  # show a pipeline report inline
    display(Markdown((LOGS / name).read_text()))

In [ ]:
# Data (all produced by src/run_pipeline.py)
panel  = pd.read_parquet(CLEAN / "zhvi_analysis_panel.parquet")        # ZIP x month: zhvi, log_zhvi, yoy_pct, index_dec2019, period, in_window
zg     = pd.read_csv(CLEAN / "zip_growth.csv", dtype={"zip": str})      # one row per ZIP: December YoY, cumulative growth, growth relative to metro
dist   = pd.read_csv(CLEAN / "zip_distance.csv", dtype={"zip": str})    # distance to city hall / nearest center / ZCTA centroid, bands
flags  = pd.read_csv(CLEAN / "zip_flags.csv", dtype={"zip": str})       # data-quality flags per ZIP
acs    = pd.read_csv(CLEAN / "zip_acs.csv", dtype={"zip": str})         # ACS 2015-2019 (_2019) and 2019-2023 (_2023) features
zori   = pd.read_parquet(CLEAN / "zori_analysis_panel.parquet")
zorig  = pd.read_csv(CLEAN / "zori_growth.csv", dtype={"zip": str})     # fixed sample: ZORI at Dec 2019 and Dec 2025
zcta   = gpd.read_file(CLEAN / "zcta_boundaries.gpkg")                  # polygons for the 1,099 ZIPs
print(panel.shape, zg.shape, zcta.shape)

In [ ]:
# Plot conventions (set once so every figure matches): fixed color per metro, one font scale, one color map per quantity.
METROS = ["Phoenix", "Austin", "Atlanta", "Dallas", "Tampa", "Charlotte", "Nashville", "Las Vegas"]
PERIODS = ["pre-2020", "2020-2022", "2023-2025"]
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10,
                     "figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False, "axes.spines.right": False})
# TODO: choose the metro palette (colorblind-safe, 8 categories) and the sequential/diverging maps; see the dataviz guidance.

import viz, viz_eda
viz.set_style()                               # shared figure style (src/viz.py)
gd = viz.gradient_data(CLEAN)                 # one row per ZIP: distances, levels, annualized growth; `balanced` = same ZIPs in every panel

## 1. Data sources, preparation, and cleaning
Five public sources, joined at ZIP / ZCTA level for eight Sun Belt metros. Full provenance (URL, release, download date, size) is in `docs/SOURCES.md`. Everything below is produced by `python src/run_pipeline.py`.

### 1.1 Sources
TODO: one short paragraph per source — what it is, why we use it, what the unit is.
- **Zillow ZHVI** (outcome): ZIP-level, all homes, mid-tier, smoothed, seasonally adjusted, monthly 2000–2026.
- **Zillow ZORI** (secondary outcome): rents, same geography, ~8,400 ZIPs nationally.
- **ACS 5-year**, 2015–2019 (pre-pandemic baseline) and 2019–2023 (current composition, descriptive only): B08006, C24010, B19013, B25024, B01003 at ZCTA level via the Census API.
- **Census Gazetteer** (ZCTA centroids), **CBSA delineation** (county crosswalk), **cartographic ZCTA boundaries** (maps).
- **City-hall coordinates**, hand-compiled with a source recorded for each of the eight metros (plus a second center for Dallas–Fort Worth and Tampa–St. Petersburg).

### 1.2 What we did to the data and what we found while doing it
Highlights to write up (each is backed by a report below): metro membership checked two ways; ZIP→ZCTA match rates; ZHVI gaps; ZORI coverage limits; a margin-of-error reliability flag on the pre-pandemic work-from-home share; two center definitions plus a second center for two-center metros; and one consolidated finding that most data problems sit at the periphery.

In [ ]:
log('zhvi_quality_report.md')

In [ ]:
log('metro_crosswalk_report.md')

In [ ]:
log('distance_report.md')

In [ ]:
log('acs_report.md')

In [ ]:
log('zori_coverage_report.md')

In [ ]:
log('boundaries_report.md')

In [ ]:
log('peripheral_gap_pattern.md')

In [ ]:
log('growth_report.md')

## 2. Exploratory analysis
Hypothesis-neutral: we describe, we do not yet test. Every figure gets a one-line takeaway under it.

### 2.1 Distributions (data quality first, then levels and growth)
- Price levels by metro and year — histograms and box plots, log scale. *Is a log transform warranted?* (Skewness table in `growth_report.md` says yes.)
- Cumulative growth by period and metro — box plots / violins. *How wide is the spread within a metro compared with the gap between metros?*
- ZHVI and ZORI coverage over time and distance — a small multiple of the data-quality flags (the figure behind Section 1).

In [ ]:
flags = pd.read_csv(CLEAN / "zip_flags.csv", dtype={"zip": str})
for name, fig in [("2_1a_distributions", viz_eda.distributions(gd)),
                  ("2_1b_log_check", viz_eda.log_check(gd)),
                  ("2_1c_flags_by_distance", viz_eda.flags_by_distance(flags))]:
    fig.savefig(FIGS / f"{name}.png"); plt.show()

**Takeaway 2.1 (DRAFT): distributions**

*What it shows.* Home values differ a lot between metros and are right-skewed within them (skewness about 2.6 in dollars, 0.4 in logs), so we plot prices on a log scale. Median ZIP values in Dec 2019 run from about $204k (Charlotte) to $331k (Austin). Every metro jumped by Dec 2022 (Austin's median reached $525k, Phoenix's $460k) and then diverged: Austin's median fell to $446k by Dec 2025 and Phoenix's to $441k, while Atlanta, Charlotte and Nashville edged up. Annualized growth in 2020–22 had medians of 12.6% (Las Vegas) to 17.5% (Tampa) a year; in 2023–25 medians ran from −5.1% (Austin) to +1.5% (Charlotte). Inside a metro the middle half of ZIPs spans a 1.4× to 1.8× range in value. The data-quality heatmap restates Section 1: beyond 30 miles, 64% of ZIPs have no rent series (6% within 10 miles) and 53% have an unreliable work-from-home share (11% within 10 miles).

*What is surprising.* The spread across ZIPs within a metro (a middle-half growth range of about 1 to 5 points a year) is as large as many of the differences between metro medians, so a metro-level average hides a lot. Pre-2020 growth was the most dispersed in Dallas and Atlanta.

*What we cannot conclude.* These are nominal values from a smoothed, model-based index, so timing is blurred and inflation is not removed. The thinning of the sample at the metro edge (see the heatmap) means the outer end of every later plot rests on fewer, less reliable ZIPs.

### 2.2 The gradient (SQ1)
- Scatter: cumulative growth against distance from the center, one panel per metro, columns = period (pre-2020, 2020–2022, 2023–2025). Fit line per panel with its slope annotated; same y-axis scale across metros.
- Slope by year (December YoY on distance), one line per metro: *does the slope change at 2020 and does it come back?*
- Same figure with the nearest-center distance for Dallas and Tampa, next to the city-hall version: *does the second center change the picture?*

In [ ]:
# `viz`, `viz_eda`, `gd` (ZIP-level gradient data) and TABLES are set up at the top of the notebook.

**Figure 2.2a — the gradient, one metro at a time.** Top row: home value (log scale) against distance at three dates. Bottom row: annualized growth for three periods. Solid line = linear fit, dashed = smooth (LOWESS) fit. Same ZIPs in every panel.

In [ ]:
for metro in viz.METRO_ORDER:
    fig = viz.gradient_metro(gd, metro)
    fig.savefig(FIGS / f"2_2_gradient_{metro.lower().replace(' ', '_')}.png"); plt.show()

**Takeaway 2.2a: the gradient, metro by metro**

*What it shows.* Home values fall with distance from the center in six of the eight metros (Phoenix, Austin, Atlanta, Tampa, Charlotte, Nashville), but the strength varies. Charlotte (about −18% per 10 miles, R² about 0.5), Austin (−15%) and Nashville (−13%) have steep, fairly tight gradients; Atlanta, Tampa and Phoenix are weaker (−4% to −9%, R² under 0.12); Dallas and Las Vegas have no linear gradient (R² below 0.01). The prediction was a flattening in 2020–22 that then holds. Only Atlanta follows it and keeps going: its level slope moves from −7.7% to −6.0% to −3.0% and its growth slope from −0.44 to +0.69 to +1.03 points per year per 10 miles (the growth-slope change is clear; the level-slope intervals overlap and the change was not tested). Austin's growth slope is positive only in 2023–25 (+0.63), though not clearly different from its 2020–22 value (+0.21, interval −0.22 to +0.64). Tampa's growth slope is +0.57 in 2020–22 and −0.32 in 2023–25 (interval −0.68 to +0.03), so its flattening did not hold. Phoenix's growth slope falls from +0.46 to −0.30; its level slope (−4.3% and −5.6%) has overlapping intervals, and its gradient disappears when the 16 far-out ZIPs with unreliable exposure estimates are dropped. Charlotte shows no detectable change, and Nashville's outer ZIPs were already outgrowing the center before 2020 (+1.02).

*What is surprising.* In Phoenix and Las Vegas the smooth curves are humped (peaks about 24 and 15 miles from city hall; a quadratic term is significant, p = 0.004 and p < 0.001 for Dec 2025). But the Phoenix hump rests on sparse far-out ZIPs: dropping the 20 ZIPs beyond 40 miles gives a level slope of +8.1% per 10 miles (interval +1.4 to +15.1), so inside 40 miles values rise with distance.

*What we cannot conclude.* By a rough rule (the largest shift between periods exceeds the width of one interval; a comparison, not a test), the growth slopes of Atlanta, Nashville, Tampa, Phoenix and, marginally, Austin shifted; Charlotte, Dallas and Las Vegas did not. Nothing here says remote work caused any of it, and we have no dense metros to compare with.

**Read with care.** (1) The intervals are plain OLS and ignore that nearby ZIPs are correlated, so they are probably too narrow. (2) These are simple fits, not the planned event study, with no controls beyond demeaning by metro (and, in 2.4, pre-2020 growth and income). (3) The smooth curves are drawn only over the range holding 95% of the ZIPs and rest on few ZIPs at the far end.

**Figure 2.2b — the two-center test (Dallas–Fort Worth, Tampa–St. Petersburg).** Row A: distance to the first-named city's hall. Row B: distance to the nearer of the two halls, one pooled fit. Row C: same distances, a separate fit for each center's ZIPs.

In [ ]:
fits = []
for metro in viz.TWO_CENTER:
    for measure in ("level", "growth"):
        fig, st = viz.two_center(gd, metro, measure=measure)
        fig.savefig(FIGS / f"2_2_two_center_{measure}_{metro.lower()}.png"); plt.show()
        fits.append(st.assign(metro=metro, measure=measure))
fits = pd.concat(fits); fits.to_csv(TABLES / "two_center_fits.csv", index=False)
fits.round(3)

**Takeaway 2.2b: does a second center change the picture (Dallas–Fort Worth, Tampa–St. Petersburg)?**

*What it shows.* Distance to the nearer of two centers does not recover a monocentric gradient. For Dallas, R² for home values stays at about 0.005 either way and the slope changes sign (−1.9% to +1.5% per 10 miles) with both intervals spanning zero; for Tampa it rises a little (0.065 to 0.080, averaged over the three dates). For growth the nearest-center version helps most in 2020–22 (R² 0.004 to 0.019 in Dallas, 0.117 to 0.163 in Tampa). Fitting each center's ZIPs separately points to why a pooled line fails in Dallas–Fort Worth: Fort Worth ZIPs rise in value with distance from Fort Worth's city hall by about +8% to +10% per 10 miles (R² about 0.10) while Dallas ZIPs are flat or slightly negative. In Dallas–Fort Worth the two cores sit at different price levels (median Dec 2019 value $221k within 10 miles of Dallas City Hall and $183k around Fort Worth's); in Tampa–St. Petersburg they do not ($220k and $227k).

*What is surprising.* The per-center growth slopes in Dallas–Fort Worth move in opposite directions (Dallas from +0.47 to −0.25, Fort Worth from −0.29 to +0.37). That is an exploratory observation from small groups, not a finding.

*What we cannot conclude.* A second center matters for the price level in Dallas–Fort Worth, not that two city halls explain the gradient; in Tampa it changes little. A city hall is only a proxy for the center of employment, and we have no job-location data. The St. Petersburg group has 39 ZIPs and a level-slope interval from about −15% to +34% per 10 miles, so it says almost nothing.

**Read with care.** (1) The intervals are plain OLS and ignore that nearby ZIPs are correlated, so they are probably too narrow. (2) These are simple fits, not the planned event study, with no controls beyond demeaning by metro (and, in 2.4, pre-2020 growth and income). (3) The smooth curves are drawn only over the range holding 95% of the ZIPs and rest on few ZIPs at the far end.

**Figure 2.2c — slope summary.** Linear slopes with 95% intervals for every metro (and the nearest-center version for Dallas and Tampa).

In [ ]:
tab = viz.slope_table(gd); tab.to_csv(TABLES / "gradient_slopes.csv", index=False)
fig = viz.slope_summary(tab); fig.savefig(FIGS / "2_2_slope_summary.png"); plt.show()
tab.round(2)

**Takeaway 2.2c: the slopes side by side**

*What it shows.* Against the prediction (flattening in 2020–22 that then holds): Atlanta fits it and keeps going; Austin's growth slope turns positive only in 2023–25; Tampa flattens and then falls back; Charlotte, Dallas and Las Vegas show no flattening; Phoenix goes the other way on growth, but its gradient depends on far-out ZIPs; and Nashville's outer-ZIP advantage predates 2020. So the hypothesis is not holding uniformly, and persistence beyond 2022 is clear only in Atlanta. Level slopes move little between Dec 2019 and Dec 2025 (within about 1.5 points everywhere except Atlanta, +4.7, and Austin, +2.2), which fits arithmetic: a growth slope of +1 point per year per 10 miles adds only about 3 points to the level slope over three years.

*What is surprising.* The metros with the steepest linear gradients (Charlotte, Austin, Nashville) show only small changes in their level slopes, while Atlanta, which starts from a modest gradient, changes most. We cannot compare with dense metros, because none are in our sample, so the plan's claim that flattening is smaller here is untested.

*What we cannot conclude.* Distance from the center is not remote-work exposure (that link is 2.4). A metro-wide change in prices, such as the 2022 mortgage-rate shock, is invisible in these within-metro slopes but could still shape them.

**Read with care.** (1) The intervals are plain OLS and ignore that nearby ZIPs are correlated, so they are probably too narrow. (2) These are simple fits, not the planned event study, with no controls beyond demeaning by metro (and, in 2.4, pre-2020 growth and income). (3) The smooth curves are drawn only over the range holding 95% of the ZIPs and rest on few ZIPs at the far end.

### 2.3 Spatial structure
- Choropleth of growth (relative to the metro median) per metro and period, one shared diverging color scale.
- Mark the center(s). *Smooth rings (monocentric) or patches (polycentric)?*

In [ ]:
import geopandas as gpd
zc = gpd.read_file(CLEAN / "zcta_boundaries.gpkg")[["zip", "geometry"]]
ref = pd.read_csv(ROOT / "data" / "reference" / "city_centers.csv")
rel = viz_eda.relative_growth(gd)
lim = viz_eda.growth_limit(rel)            # one color scale for every metro and period
for metro in viz.METRO_ORDER:
    fig = viz_eda.growth_maps(zc, rel, ref, metro, lim)
    fig.savefig(FIGS / f"2_3a_growth_map_{metro.lower().replace(' ', '_')}.png"); plt.show()
fig = viz_eda.level_quintile_maps(zc, gd, ref)
fig.savefig(FIGS / "2_3b_level_quintile_maps.png"); plt.show()

**Takeaway 2.3: maps**

*What it shows.* A direction test (permutation test on the compass bearing of the highest-value fifth, 5,000 permutations) finds the highest-value ZIPs concentrated on one side of the center in five of the eight metros, p < 0.01 in each: northeast of Phoenix's city hall, northwest of Austin's, north of Atlanta's, northeast of the two Dallas–Fort Worth halls, south of Nashville's. In Dallas–Fort Worth both cores sit in lower fifths (63% of ZIPs within 10 miles of either hall are in the bottom two fifths, against 40% expected). Tampa, Charlotte and Las Vegas show no clear direction (p = 0.77, 0.15, 0.08); in Las Vegas the highest-value fifth points southwest, and the dark northwest area on the map is a few very large ZIPs. The growth maps (relative to each metro's own median, one color scale for all) show that in Atlanta, Austin and Nashville the ZIPs within 10 miles of the center grew more slowly than the metro in 2023–25 and those beyond 30 miles faster (−2.9 against +1.5, −1.3 against +0.4, −1.7 against +0.8 points a year; p < 0.01 in each), which matches their positive 2023–25 slopes. Charlotte's core is only mildly slow (−0.7, not significantly different from its outer ZIPs) and its far-south ZIPs are also slow (−0.7 against +0.6, 10 ZIPs), which may be why its slope is near zero. Before 2020, Atlanta's ZIPs north of city hall were slow (−3.7) and those south and southeast fast (+1.1, +2.2); the Dallas and Fort Worth cores grew faster than their surroundings (+2.4 against −0.5), with the northern suburbs slower (−3.5 north against +1.4 south of the midpoint); Phoenix's far southeast was fastest (+2.6) while its inner core was only modestly above the median (+0.9).

*What is surprising.* A wedge is not the shape the monocentric model predicts, which would help explain why one distance variable fits poorly where the wedge is strongest (Phoenix, Dallas); but Austin and Nashville have wedges and still fit moderately well, so the wedge alone does not account for the fit. In seven of eight metros (all but Austin) growth varied more across ZIPs before 2020 than in 2020–22, so the post-2020 maps are, if anything, quieter; but the pre-2020 window is two years long, which by itself inflates annualized dispersion.

*What we cannot conclude.* The maps are descriptive: growth is relative to the metro median, large thinly populated ZIPs dominate the area (Las Vegas is hard to read), white areas are not in our ZIP set, and we have not measured spatial clustering or explained why the wedges form.



### 2.4 Relationships (SQ2)
- Pre-pandemic work-from-home share (2015–2019) against 2019–2022 growth: binned scatter, pooled and within metro (demeaned). Show the unreliable-share ZIPs in a different marker.
- Correlation matrix of exposure, income, housing mix, distance, growth. *How much of the exposure signal is just income and distance?*

In [ ]:
acs = pd.read_csv(CLEAN / "zip_acs.csv", dtype={"zip": str})
xd = viz_eda.exposure_data(gd, acs)          # same ZIPs as 2.2, with a 2015-2019 work-from-home share
fig, slopes = viz_eda.exposure_scatter(xd)
fig.savefig(FIGS / "2_4a_exposure_scatter.png"); plt.show()
slopes.to_csv(TABLES / "exposure_slopes.csv", index=False)
fig = viz_eda.exposure_confounds(xd); fig.savefig(FIGS / "2_4c_exposure_confounds.png"); plt.show()
fig, cmat = viz_eda.exposure_corr(xd); fig.savefig(FIGS / "2_4b_exposure_corr.png"); plt.show()
cmat.to_csv(TABLES / "exposure_corr.csv"); slopes.round(3)

**Takeaway 2.4: pre-pandemic work-from-home share and growth**

*What it shows.* High-work-from-home and high-income ZIPs overlap strongly. Within a metro the work-from-home share has a rank correlation of +0.66 with income, +0.74 with home value and +0.74 with the management and professional share; more expensive ZIPs tended to grow more slowly before 2020 (rank correlation −0.72). So the pre-2020 relationship between exposure and growth is clearly negative (−1.48 points a year per +5 percentage points of work-from-home share, R² 0.13); within metros the slope is −0.30 in 2020–22 and +0.40 in 2023–25 (pooled +0.17 and −0.10). **Controlling for pre-2020 growth reduces the later slopes to +0.03 (interval −0.13 to +0.20) and +0.13 (−0.03 to +0.30); adding income gives −0.15 (−0.36 to +0.06) and 0.00 (−0.15 to +0.15). All four intervals include zero.** The later slopes are largely accounted for by the earlier trend, and we find no relationship with exposure that is separate from it.

*What is surprising.* The strongest relationship in the figure is the pre-2020 placebo, and the plan's predicted positive relationship in 2020–22 does not appear within metros. The slope does rise across the three periods, but its only positive value is in 2023–25 (the plan expected the strongest relationship in 2021–22) and it falls to about zero once pre-2020 growth is controlled, so we do not read it as support for the hypothesis. Dropping the 233 ZIPs in this sample with an unreliable estimate barely changes the slopes (−1.41, −0.26, +0.52), though those ZIPs scatter somewhat more widely (residual spread 8% to 24% larger).

*What we cannot conclude.* The controls handle only pre-2020 growth and income (an exploratory regression, not the event study); exposure also tracks occupation mix, and we have no education measure. Nothing here says remote work did or did not affect prices. About 22% of ZIPs have an unreliable estimate, concentrated in the outer ring.

**Read with care.** (1) The intervals are plain OLS and ignore that nearby ZIPs are correlated, so they are probably too narrow. (2) These are simple fits, not the planned event study, with no controls beyond demeaning by metro (and, in 2.4, pre-2020 growth and income). (3) The smooth curves are drawn only over the range holding 95% of the ZIPs and rest on few ZIPs at the far end.

### 2.5 Trajectories (SQ3, exploratory)
- Index (Dec 2019 = 100) paths for all ZIPs as faint lines with metro medians on top; faceted by distance band.
- Optional: k-means on normalized paths with a silhouette plot; map the clusters. *Are the groups distinct or a continuum?*

In [ ]:
panel = pd.read_parquet(CLEAN / "zhvi_analysis_panel.parquet", columns=["zip", "date", "index_dec2019"])
P = viz_eda.path_matrix(panel, gd)           # ZIPs with a full monthly series, index Dec 2019 = 100
fig = viz_eda.trajectory_panels(P, gd); fig.savefig(FIGS / "2_5a_trajectories.png"); plt.show()
fig = viz_eda.relative_paths(P, gd);    fig.savefig(FIGS / "2_5b_relative_paths.png"); plt.show()
sil, labels, k = viz_eda.cluster_paths(P)    # k-means on the log index path; k = best silhouette among 3 to 5
fig, comp = viz_eda.cluster_figure(P, gd, sil, labels, k); fig.savefig(FIGS / "2_5c_clusters.png"); plt.show()
profile = viz_eda.cluster_profile(gd, acs, labels)
profile.round(2).to_csv(TABLES / "cluster_profile.csv"); sil.round(3).to_csv(TABLES / "cluster_silhouette.csv")
profile.round(2)

**Takeaway 2.5: price paths and groups of ZIPs**

*What it shows.* Every metro followed the same broad shape: a steep rise from 2020 to 2022 (metro median ZIP indexes of about 143 to 162 by Dec 2022, Dec 2019 = 100) and then a flat or falling stretch, with Austin falling furthest. Within most metros the distance bands move together (in Phoenix, Dallas and Charlotte each band's median stays within about 4% of the metro). Atlanta is the one clear case of inner ZIPs losing ground to outer ones: its inner band (0–10 miles) fell to 85% of the metro's median index by Dec 2022 and 83% by Dec 2025 (bootstrap 95% interval 80.5 to 87.2) while the 30+ mile band rose to about 105% (103.6 to 106.2). That is the pattern the literature calls a donut, but the inner decline began before 2020 (103.3 in Jan 2018 to 100 at Dec 2019), and the post-2020 fall is much steeper (15 points in three years against 3 in two). Nashville's inner decline also predates 2020 (107.2 to 100), and Austin's outer band interval spans 100. Tampa does not fit: its inner and outer bands both gained on the middle bands (105.8 and 106.7 against 97.7 and 98.8). Grouping the 1,072 ZIPs with complete series by path shape (k-means on the log index) gives four groups: Group 3 (89 ZIPs, 74% in Austin) shows a boom and a correction; Group 4 (48 ZIPs, median 4 miles from city hall, 18% single-family) rose least; Groups 1 and 2 differ mainly in amplitude.

*What is surprising.* The groups follow metro more than distance (Cramér's V 0.60 for metro, 0.18 for distance band); housing type separates only Group 4. We find no delayed-appreciation group, but clustering on the log index is dominated by overall amplitude (the first principal component explains 71% of path variance), so shape types may not separate; shape-only clustering was not tried.

*What we cannot conclude.* The groups are only weakly separated: silhouette 0.22 to 0.38 across k = 2 to 8 (0.36 at k = 4, no better than k = 2 at 0.375), 25% of ZIPs below 0.25, stable to the starting point but not always to resampling (adjusted Rand index median 0.85, fifth percentile 0.45). We cannot say four is the right number or that the groups are real types and not a partition of a continuum. The 27 ZIPs with incomplete series are left out, and several band medians rest on 4 to 15 ZIPs.



### 2.6 Robustness views
- Full sample vs. flag-free sample (see `zip_flags.csv`), side by side.
- ZORI check on the fixed ZIP sample, stated as an inner/middle-metro check (see the ZORI limitation in Section 1).

In [ ]:
flags = pd.read_csv(CLEAN / "zip_flags.csv", dtype={"zip": str})
zorig = pd.read_csv(CLEAN / "zori_growth.csv", dtype={"zip": str})
tab = viz_eda.robustness_table(gd, flags)        # gradient slopes: all ZIPs vs flag-free ZIPs
tab.to_csv(TABLES / "robustness_flag_free.csv", index=False)
fig = viz_eda.robustness_figure(tab); fig.savefig(FIGS / "2_6a_flag_free.png"); plt.show()
# center definition: distance to the ZCTA centroid of the city-hall ZIP instead of city hall itself
dz = pd.read_csv(CLEAN / "zip_distance.csv", dtype={"zip": str})[["zip", "dist_zctactr_mi"]]
tab2 = viz_eda.robustness_table(gd.merge(dz, on="zip"), flags, distance="dist_zctactr_mi")
cmp_ = tab[tab["sample"] == "all ZIPs"].merge(tab2[tab2["sample"] == "all ZIPs"], on=["metro", "measure", "panel"],
                                              suffixes=("_cityhall", "_zctactr"))
cmp_["diff"] = cmp_["slope_zctactr"] - cmp_["slope_cityhall"]
cmp_.round(3).to_csv(TABLES / "robustness_center_definition.csv", index=False)
print(cmp_.groupby("measure")["diff"].agg(lambda s: s.abs().max()).round(3).to_dict(), "= largest slope change from the ZCTA-centroid distance")
fig, rent = viz_eda.rent_vs_price(gd, zorig); fig.savefig(FIGS / "2_6c_rent_vs_price.png"); plt.show()
rent.to_csv(TABLES / "rent_vs_price_slopes.csv", index=False); rent.round(2)

In [ ]:
# Extra checks behind the interpretive claims in 2.2 to 2.6 (written by src/audit_checks.py; run it to regenerate).
from IPython.display import display
AUD = ROOT / "outputs" / "tables"
print("Direction test for the highest-value fifth (Table 4 in the report):")
display(pd.read_csv(AUD / "audit_wedge.csv"))
print("Exposure slope with pre-2020 growth and income controlled (Table 6):")
display(pd.read_csv(AUD / "audit_exposure_controls.csv"))
print("Inner and outer distance-band index relative to the metro, with bootstrap intervals (Table 7):")
display(pd.read_csv(AUD / "audit_band_bootstrap.csv").query("band in ['0-10 mi', '30+ mi']"))
print("Phoenix level slope under different exclusions:")
display(pd.read_csv(AUD / "audit_phoenix_exclusions.csv"))

**Takeaway 2.6: robustness**

*What it shows.* Dropping the flagged ZIPs (834 of 1,082 remain) changes the picture more for price levels than for growth. Level slopes move by up to about 6 points per 10 miles (flatter on average by about 2), consistent with the flagged ZIPs being cheaper (median $213k against $269k) and farther out. Growth slopes change by 0.00 on average (range −0.37 to +0.59). Phoenix matters: with flagged ZIPs removed its level slope goes from −4.3% to +0.5% per 10 miles and all three growth slopes are near zero. Only the work-from-home flag does this (16 ZIPs, median distance 56 miles); the other three flags change nothing, and dropping the 20 ZIPs beyond 40 miles gives +8.1% (+1.4 to +15.1). So Phoenix's negative gradient comes from far-out ZIPs. Atlanta's pattern survives (+1.25 in 2020–22 against +0.69, overlapping intervals); Nashville's and Tampa's growth slopes hold; Dallas's 2023–25 slope is −0.35 (−0.57 to −0.12) without flagged ZIPs against −0.03 with all, so the Dallas result depends on the sample. The center definition does not matter (no slope changes by more than 0.5 percentage points or 0.06 points per year). On the 478 ZIPs with rents at both ends, price growth rose with distance in 2020–22 (+0.83) and so did rent growth (+0.41); in 2023–25 price growth no longer rose with distance (−0.07) while rent growth still did (+0.33).

*What is surprising.* How much one metro depends on its fringe: Phoenix goes from a steepening gradient to none. The rent result is the kind of contrast the rent check was designed to look for, since rents do not depend directly on mortgage financing.

*What we cannot conclude.* That interest rates explain the 2023–25 stall in prices: rents are smoothed and lag prices, so they may still be adjusting to the 2020–22 shift; the rent comparison covers only 41 ZIPs 30 or more miles out; and ZIPs with rents are larger and more multifamily than the rest. The flag-free subset is a selected sample, neither more nor less correct than the full one.

**Read with care.** (1) The intervals are plain OLS and ignore that nearby ZIPs are correlated, so they are probably too narrow. (2) These are simple fits, not the planned event study, with no controls beyond demeaning by metro (and, in 2.4, pre-2020 growth and income). (3) The smooth curves are drawn only over the range holding 95% of the ZIPs and rest on few ZIPs at the far end.

## 3. Implications
**Does the hypothesis hold?** Not uniformly. The expected flattening of the price gradient in 2020–22 that then persists is clear only in Atlanta; Austin's growth slope turns positive only in 2023–25, and Tampa's flattening does not hold. Charlotte, Dallas and Las Vegas show no flattening, Phoenix's negative gradient depends on far-out ZIPs, and Nashville's outer-ZIP advantage predates 2020. The prediction for sub-question 2 (more pre-pandemic work-from-home potential, more growth, strongest in 2021–22) is not what we see: within metros the relationship is negative until 2023–25, and it falls to about zero once pre-2020 growth is controlled, so it largely reflects an existing pre-2020 trend that is correlated with income. Sub-question 3's groups are only weakly separated, partly track metro, and are dominated by one dimension of overall amplitude.

**What this means for the modeling.** A single linear distance term describes these metros poorly: expensive ZIPs are concentrated on one side of the center in five of the eight metros, values hump in Phoenix and Las Vegas, the two cores of Dallas–Fort Worth sit at different price levels, and Dallas and Las Vegas show no linear gradient. The models should therefore allow a smooth distance effect and center-specific terms, and the job-center data we set aside is worth bringing back. Because pre-2020 growth is already clearly related to work-from-home share, and controlling for it removes the later relationship, the event study should include pre-period growth and income and report the pre-2020 coefficients. Results that depend on the outer ZIPs (Phoenix, Dallas) should be shown with and without the flagged ZIPs, with standard errors clustered for spatial correlation, and treated as provisional. Rents can check the rate-shock explanation only for the inner and middle metro, on a fixed set of ZIPs, and because ZORI is smoothed a lag between rents and prices cannot be ruled out; the plan's metro-by-month fixed effects remove metro-wide movements such as the 2022 rate shock.

**What the data cannot answer.** Whether remote work caused any of this; what happened in the outer ring, where rents and reliable exposure measures are thin; and whether other centers explain the wedges. **Next.** Add the job-center data, build the event-study specification with pre-trend tests and spatially clustered errors, and repeat the grouping within metros so that group membership is not just metro.

## References
Data sources only; full provenance (URLs, vintages, sizes, retrieval dates) is in `docs/SOURCES.md`.

1. Zillow Group. Zillow Home Value Index (ZHVI), ZIP code, all homes, mid-tier, smoothed, seasonally adjusted. Zillow Research, data through August 2026. https://www.zillow.com/research/data/ (retrieved 6 Oct 2026).
2. Zillow Group. Zillow Observed Rent Index (ZORI), ZIP code, all homes plus multifamily, smoothed. Zillow Research, data through August 2026. https://www.zillow.com/research/data/ (retrieved 6 Oct 2026).
3. U.S. Census Bureau. American Community Survey 5-Year Estimates, 2015–2019, tables B08006, C24010, B19013, B25024, B01003, ZIP Code Tabulation Areas. Census Data API. https://www.census.gov/data/developers/data-sets/acs-5year.html (retrieved 6 Oct 2026).
4. U.S. Census Bureau. American Community Survey 5-Year Estimates, 2019–2023, same tables and geography. Census Data API (retrieved 6 Oct 2026).
5. U.S. Census Bureau. 2024 Gazetteer Files, ZIP Code Tabulation Areas. https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (retrieved 6 Oct 2026).
6. U.S. Census Bureau. Core Based Statistical Areas, Metropolitan Divisions, and Combined Statistical Areas, July 2023 (List 1). https://www2.census.gov/programs-surveys/metro-micro/geographies/reference-files/2023/delineation-files/list1_2023.xlsx (retrieved 6 Oct 2026).
7. U.S. Census Bureau. Cartographic Boundary File, 2020 ZIP Code Tabulation Areas, 1:500,000. https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_zcta520_500k.zip (retrieved 6 Oct 2026).
8. Wikipedia contributors. City hall articles for Phoenix, Austin, Atlanta, Dallas, Tampa, Nashville (Davidson County Courthouse) and Las Vegas, for building coordinates. https://en.wikipedia.org (retrieved 6 Oct 2026).
9. OpenStreetMap contributors. Nominatim address search, for the Charlotte, Fort Worth and St. Petersburg city-hall points and for coordinate cross-checks. https://nominatim.openstreetmap.org (retrieved 6 Oct 2026).